In [50]:
import pandas as pd
from sqlalchemy import create_engine
import urllib
import ast
from datetime import timedelta
import os
from dotenv import load_dotenv

load_dotenv()

True

In [51]:
params = urllib.parse.quote_plus(
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={os.getenv('DB_SERVER')};"
    "DATABASE=Ventas_Comerssia;"
    f"UID={os.getenv('DB_USER')};"
    f"PWD={os.getenv('DB_PASSWORD')};"
)


# Crear el motor de conexión
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

In [52]:
# query_com = """
# SELECT DISTINCT Pedido,
#     c.Cliente,
#     c.Nombres + ' ' + c.Apellidos AS Nombre,
#     c.Email AS Email,
#     c.Celular AS Celular,
#     v.Fecha AS Fecha_Venta,
#     v.Canal,
#     SUM(v.Venta_Neta) AS Venta
# FROM Ventas_Comerssia.dbo.Contacto_Clientes c
# INNER JOIN Ventas_Comerssia.dbo.Ventas_Unificadas v ON c.Cliente = v.Cliente
# WHERE v.fecha >= '2026-06-20'
#     AND v.Fecha <= '2026-06-20'
#     AND Canal IN ('Shopify')
#     AND v.Venta_Neta > 0
# GROUP BY  Pedido,
#     c.Cliente,
#     c.Documento,
#     c.Nombres + ' ' + c.Apellidos,
#     c.Email,
#     c.Celular,
#     v.Fecha,
#     v.Canal
# """

# # Ejecutar y cargar en DataFrame
# df_ventas = pd.read_sql(query_com, engine)

# df_ventas.head()

In [53]:
# CONSULTAS CUMPLEAÑOS

query_data = """
SELECT DISTINCT Pedido,
    v.Cliente, 
	cc.Nombres + ' ' +  cc.Apellidos AS Nombre, 
	cc.Email, cc.Celular, 
	v.Fecha AS Fecha_Venta,
	Canal,
    SUM(Venta_Neta) AS Venta 
FROM Ventas_Comerssia.dbo.Ventas_Unificadas v
INNER JOIN Ventas_Comerssia.dbo.Contacto_Clientes cc ON cc.Cliente = v.Cliente
WHERE Codigo_Descuento IN ('0222 Reactivación Cumpleaños Enero-Mayo Manual','CELEBRA25')
	AND Fecha BETWEEN '2026-06-30' AND '2026-06-30'
    AND Canal = 'Shopify'
GROUP BY  Pedido,
    v.Cliente,
    cc.Nombres + ' ' +  cc.Apellidos,
    cc.Email,
    cc.Celular,
    v.Fecha,
    v.Canal
"""


# Ejecutar y cargar en DataFrame
df_ventas = pd.read_sql(query_data, engine)
df_ventas.head()

,Pedido,Cliente,Nombre,Email,Celular,Fecha_Venta,Canal,Venta
0,100961,C43842111,DIANA CASTILLO RIVERA,dianacasri@gmail.com,3007004307,2026-06-30,Shopify,436115.70
1,100962,C42077634,MARIA GARCIA,costygarcia@gmail.com,3106139246,2026-06-30,Shopify,151254.00
2,100967,C43203801,MARCELA VÁSQUEZ,marcevg38@gmail.com,3105250491,2026-06-30,Shopify,207974.25
3,100968,C35260487,GABRIELA PINILLOS,gacapima@gmail.com,1111111111,2026-06-30,Shopify,83189.70
4,100970,C1094903855,NATALIA MEDINA,mapi16_61@hotmail.com,3146319438,2026-06-30,Shopify,112180.05


In [54]:
# Leer archivo
df_revie = pd.read_excel("Atribucion.xlsx")

# Seleccionar columnas relevantes
df_campania = df_revie[['Email', 'First Name', 'Last Name', 'Phone', 'Company']].copy()

# 4. Normalizar correos
df_campania['Email'] = df_campania['Email'].str.lower().str.strip()


In [55]:
# Normalizar df_ventas
df_ventas['Email'] = df_ventas['Email'].str.lower().str.strip()
df_ventas['Fecha_Venta'] = pd.to_datetime(df_ventas['Fecha_Venta']).dt.date

# Merge por email
df_atribucion = pd.merge(df_ventas, df_campania, on='Email', how='inner')

# Sumar la venta total atribuida
total_venta = df_atribucion['Venta'].sum()

filas = len(df_atribucion) # Ver cuántas filas hay realmente

print(f"Total de Ventas: {filas}")
print(f"Total de venta atribuida: {total_venta:,.2f}"  )
# Resultado final
df_atribucion.head()

Total de Ventas: 9
Total de venta atribuida: 1,934,160.54


,Pedido,Cliente,Nombre,Email,Celular,Fecha_Venta,Canal,Venta,First Name,Last Name,Phone,Company
0,100961,C43842111,DIANA CASTILLO RIVERA,dianacasri@gmail.com,3007004307,2026-06-30,Shopify,436115.70,DIANA,CASTILLO RIVERA,3007004307,43842111
1,100962,C42077634,MARIA GARCIA,costygarcia@gmail.com,3106139246,2026-06-30,Shopify,151254.00,MARIA,GARCIA,3106139246,42077634
2,100967,C43203801,MARCELA VÁSQUEZ,marcevg38@gmail.com,3105250491,2026-06-30,Shopify,207974.25,MARCELA,VÁSQUEZ,3105250491,43203801
3,100968,C35260487,GABRIELA PINILLOS,gacapima@gmail.com,1111111111,2026-06-30,Shopify,83189.70,GONZALO ANDRES,TORRES BETANCOURT,3153491526,3805782
4,100970,C1094903855,NATALIA MEDINA,mapi16_61@hotmail.com,3146319438,2026-06-30,Shopify,112180.05,NATALIA,MEDINA,3146319438,1094903855


In [56]:
df_atribucion.to_excel('Ventas.xlsx', index=False)